# Execute and inspect the declared 3PTB two-search demonstration

**Run All executes two new Vina searches**, one rigid/native and one
rigid/displaced, after replaying the explicit preparation notebook once.
It then saves and inspects both complete results. Use a fresh output directory;
there are no retries, tuning, fixed-pose rescoring, RMSD or viewer actions.
This is a small workflow demonstration, not the old 24-search benchmark or an
estimate of experimental affinity, recovery or chemical-model suitability.

Use the existing five preparation inputs described in the
[preparation guide](installed_3ptb_preparation.md), plus these explicit inputs:
`BINDINGMT_3PTB_PREPARATION_NOTEBOOK`, `BINDINGMT_3PTB_PREPARATION_NOTEBOOK_SHA256`,
`BINDINGMT_RESULT_INSPECTION_NOTEBOOK`, `BINDINGMT_RESULT_INSPECTION_NOTEBOOK_SHA256`
and `BINDINGMT_3PTB_EXECUTION_OUTPUT`. Notebook identities must come from trusted
copies. They authorize the code being executed, not merely data loading.
Jupyter/IPython executes the two companion notebooks through its public `%run`.
No repository, `devtools` import, data harness or SDK is required.

The [prospective registration](installed_3ptb_execution_registration.md) defines
this exact finite population and failure policy before any search. Full thirteen-
atom poses remain intact. The nine-heavy-atom reference is not compared by RMSD
in this demonstration; a later evaluation must declare its population and
provider-owned correspondence before measuring. Molecular work stays in MolSysMT.

In [ ]:
import hashlib
import importlib.metadata as metadata
import json
import os
import shlex
import traceback
from copy import deepcopy
from pathlib import Path

import pyunitwizard as puw
from IPython import get_ipython

import bindingmt as dmt

preparation_notebook = Path(os.environ['BINDINGMT_3PTB_PREPARATION_NOTEBOOK']).resolve()
inspection_notebook = Path(os.environ['BINDINGMT_RESULT_INSPECTION_NOTEBOOK']).resolve()
companion_identities = {}
for role, notebook_path, variable in (
    ('preparation', preparation_notebook, 'BINDINGMT_3PTB_PREPARATION_NOTEBOOK_SHA256'),
    ('inspection', inspection_notebook, 'BINDINGMT_RESULT_INSPECTION_NOTEBOOK_SHA256'),
):
    expected = os.environ[variable].lower()
    if len(expected) != 64 or any(c not in '0123456789abcdef' for c in expected):
        raise ValueError('Supply externally expected companion notebook identities.')
    if hashlib.sha256(notebook_path.read_bytes()).hexdigest() != expected:
        raise ValueError('Companion notebook bytes differ from the trusted identity.')
    companion_identities[role] = expected
execution_directory = Path(os.environ['BINDINGMT_3PTB_EXECUTION_OUTPUT']).resolve()
if execution_directory.exists():
    raise FileExistsError('Use a fresh execution output directory.')
assert get_ipython() is not None, 'Run this notebook in a Jupyter/IPython kernel.'
print(
    {'scope': 'two new searches, no RMSD or viewer', 'companions': companion_identities}
)

## Replay the explicit preparation once

The trusted preparation notebook's nine unchanged cells verify source/declaration
identities, use the caller-declared maps and public provider tools, check the
original prepared controls and admit the two planned problems. Its report still
truthfully says that its own preparation-only scope executed no search.
The execution evidence below is a separate record. Existing outputs are refused.

In [ ]:
get_ipython().run_line_magic('run', shlex.quote(str(preparation_notebook)))
preparation_exports = get_ipython().user_ns
preparation_receipt = deepcopy(preparation_exports['receipt'])
problems = preparation_exports['problems']
protocol = preparation_exports['protocol']
prepared_receptor = preparation_exports['prepared_receptor']
prepared_partner = preparation_exports['prepared_partner']
assert preparation_receipt['execution_status'] == 'not_executed'
assert list(problems) == ['native', 'displaced']
assert (prepared_receptor.n_atoms, prepared_partner.n_atoms) == (2011, 13)
execution_policy = puw.configure.report()
execution_directory.mkdir(parents=True, exist_ok=False)

## Preserve the planned denominator before attempting either case

Both rows use the same admitted inputs, seed 7, exhaustiveness 1, CPU 1,
at most nine returned poses and a 3 kcal/mol retrieval window. Only the explicit
box centre differs (+60 angstrom x for the control). Actual submitted bytes and
native resolved calls belong to each completed result. Progress is append-only;
a search exception is retained and does not trigger a retry or remove a row.

In [ ]:
planned_cases = ['native', 'displaced']
execution_rows = [
    {'case': name, 'status': 'planned', 'attempts': 0, 'returned_poses': None}
    for name in planned_cases
]
with (execution_directory / 'registration.json').open('x') as stream:
    json.dump(
        {
            'schema': 'bindingmt.3ptb_two_search_registration@1',
            'cases': planned_cases,
            'maximum_searches': 2,
            'maximum_returned_poses': 18,
            'retries': 0,
            'RMSD_evaluations': 0,
            'companions': companion_identities,
            'preparation_receipt': preparation_receipt,
            'protocol': protocol.to_dict(),
            'planned_rows': execution_rows,
        },
        stream,
        indent=2,
        allow_nan=False,
    )
progress_path = execution_directory / 'progress.jsonl'
progress_path.touch(exist_ok=False)


def retain_event(event):
    with progress_path.open('a') as stream:
        stream.write(json.dumps(event, allow_nan=False) + '\n')
        stream.flush()
        os.fsync(stream.fileno())


saved_results = {}
completed_results = {}
print({'planned_cases': planned_cases, 'maximum_searches': 2, 'retries': 0})

In [ ]:
for row in execution_rows:
    case_name = row['case']
    row.update(status='attempting', attempts=1)
    retain_event(deepcopy(row))
    try:
        completed = dmt.dock(problems[case_name], protocol=protocol)
        completed_results[case_name] = completed
        record_bytes = json.dumps(completed.to_dict(), allow_nan=False).encode()
        filename = case_name + '_result.json'
        with (execution_directory / filename).open('xb') as stream:
            stream.write(record_bytes)
        saved_results[case_name] = {
            'filename': filename,
            'bytes': len(record_bytes),
            'sha256': hashlib.sha256(record_bytes).hexdigest(),
        }
        row.update(status='returned', returned_poses=len(completed.poses))
        assert len(completed.poses) <= 9
        assert all(pose.n_atoms == 13 for pose in completed.poses)
        row['result_file'] = saved_results[case_name]
    except Exception as error:
        row.update(status='failed', error_type=type(error).__name__)
        with (execution_directory / (case_name + '_failure.txt')).open('x') as stream:
            stream.write(traceback.format_exc())
    finally:
        retain_event(deepcopy(row))
        print(
            {key: row[key] for key in ('case', 'status', 'attempts', 'returned_poses')}
        )
with (execution_directory / 'outcomes.json').open('x') as stream:
    json.dump(execution_rows, stream, indent=2, allow_nan=False)
if any(row['status'] != 'returned' for row in execution_rows):
    raise RuntimeError(
        'A registered case failed; retain the evidence and do not retry.'
    )
assert puw.configure.report() == execution_policy

## Inspect the saved records without executing another search

Each result file is bound to the just-recorded producer digest before the trusted
inspection notebook runs. This is linked new-run evidence, not an independent
historical authentication. Its eight unchanged cells audit schema/provenance,
verify captured inputs, preserve scores/units/order, refuse unsupported schema
and changed captures, test nondefault units and independent detached roundtrip.
They do not reconstruct the molecular problem or compute RMSD.

In [ ]:
inspection_receipts = {}
for case_name in planned_cases:
    saved = saved_results[case_name]
    os.environ['BINDINGMT_RESULT_PATH'] = str(execution_directory / saved['filename'])
    os.environ['BINDINGMT_RESULT_SHA256'] = saved['sha256']
    get_ipython().run_line_magic('run', shlex.quote(str(inspection_notebook)))
    inspection_exports = get_ipython().user_ns
    receipt = inspection_exports['receipt']
    inspection_receipts[case_name] = deepcopy(receipt)
    assert receipt['record_audit_status'] == 'consistent'
    assert receipt['captured_bytes_verified']
    assert receipt['pose_count'] == len(completed_results[case_name].poses)
    assert inspection_exports['result'].problem is None
    artifacts = inspection_exports['record']['provenance']['backend_artifacts']
    for role in ('receptor', 'partner'):
        assert (
            artifacts[role]['sha256']
            == preparation_receipt['artifacts'][role + '.pdbqt']['sha256']
        )
assert puw.configure.report() == execution_policy

In [ ]:
execution_receipt = {
    'schema': 'bindingmt.installed_3ptb_two_search@1',
    'scope': 'two_search_workflow_and_detached_inspection',
    'companions': companion_identities,
    'versions': {
        **preparation_receipt['reader_versions'],
        'vina': metadata.version('vina'),
    },
    'source_file_sha256': preparation_receipt['source_file_sha256'],
    'declaration_file_sha256': preparation_receipt['declaration_file_sha256'],
    'cases': execution_rows,
    'inspection_receipts': inspection_receipts,
    'pose_rows': {
        name: [
            {
                'position': index + 1,
                'pose_id': pose.pose_id,
                'rank': pose.rank,
                'n_atoms': pose.n_atoms,
                'scores': deepcopy(pose.scores),
            }
            for index, pose in enumerate(completed_results[name].poses)
        ]
        for name in planned_cases
    },
    'new_searches': 2,
    'retries': 0,
    'fixed_pose_rescoring': False,
    'RMSD_evaluations': 0,
    'recovery_or_affinity_claim': False,
    'chemical_model_suitability': 'unassessed',
    'unit_policy_preserved': True,
    'native_molecular_snapshot_created': False,
    'browser_or_C6_acceptance': False,
}
with (execution_directory / 'execution_receipt.json').open('x') as stream:
    json.dump(execution_receipt, stream, indent=2, allow_nan=False)
print(json.dumps(execution_receipt, indent=2, allow_nan=False))

## What this completes

The two planned cases and every failure remain in the denominator. Returned pose
count/order/rank and recorded score descriptors retain their actual meanings.
A completed search is not proof of near-reference recovery or chemical validity.
The full records retain captured inputs, native energy vectors and resolved calls;
keep their original file hashes and producer with any later analysis.

This is separate from both the original 24-search population and the original
five-pose human result. Do not relabel those historical producers or add these
cases to their denominators. No provider wheel, public release, molecular
snapshot, visual review or general workflow acceptance follows from this example.
Before later RMSD, declare the comparison population and use public MolSysMT
selection/correspondence/geometry with BindingMT's explicit evaluation semantics.